In [14]:
# ============================================================
# STEP 1: HARVEST — Adagrasib, Sotorasib, Divarasib
# Query ClinicalTrials.gov v2 separately per drug, then combine.
# Querying 3 times separately (vs. one combined OR query) gives us
# cleaner per-drug tracking before we deduplicate in Step 2.
# ============================================================

import requests
import pandas as pd

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

# Each drug gets its own query. Statuses: Completed, Terminated,
# and Active, not recruiting — per Kiefer's filter requirement.
# ClinicalTrials.gov's actual enum values for these statuses are:
TARGET_STATUSES = "COMPLETED|TERMINATED|ACTIVE_NOT_RECRUITING"

DRUGS = ["Adagrasib", "Sotorasib", "Divarasib"]

def fetch_trials_for_drug(drug_name, page_size=100):
    """
    Pulls all studies for a single drug across our 3 target statuses.
    Returns the raw 'studies' list from the API response.
    """
    params = {
        "query.intr": drug_name,
        "filter.overallStatus": TARGET_STATUSES,
        "pageSize": page_size,
        "format": "json"
    }
    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        print(f"⚠️ Failed for {drug_name}: status {response.status_code}")
        return []

    data = response.json()
    studies = data.get("studies", [])
    print(f"✅ {drug_name}: {len(studies)} studies retrieved")
    return studies


# Run all 3 queries and tag each result with which query found it
# (useful later — a combination trial might get pulled by more than
# one drug query, which is exactly what Step 2's dedup needs to handle)
all_raw_studies = []
for drug in DRUGS:
    studies = fetch_trials_for_drug(drug)
    for s in studies:
        s["_queried_via"] = drug  # tag for traceability before dedup
    all_raw_studies.extend(studies)

print(f"\nTotal raw studies across all 3 queries (before dedup): {len(all_raw_studies)}")

✅ Adagrasib: 22 studies retrieved
✅ Sotorasib: 36 studies retrieved
✅ Divarasib: 7 studies retrieved

Total raw studies across all 3 queries (before dedup): 65


In [15]:
# ============================================================
# COMBINED STEP 2 & 3 (V2-PROD): DEEP AGGRESSIVE PARSER
# Engineered to handle nested arrays and dynamic schemas
# of the ClinicalTrials.gov API v2 accurately.
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# CONFIG — Advisor-aligned threshold placeholders
# ------------------------------------------------------------
PHASE1_AE_GRADE34_THRESHOLD = 0.40
PHASE3_MIN_ENROLLMENT = 200
PHASE3_HR_EXCELLENT_THRESHOLD = 0.60

# ------------------------------------------------------------
# SCORING ROUTINES
# ------------------------------------------------------------
def score_phase1(row):
    score = 100
    notes = ["Sample size not penalized — small n is standard for Phase 1 safety trials"]
    grade34_ae_pct = row.get("Grade34_AE_Pct")
    if grade34_ae_pct is not None and not pd.isna(grade34_ae_pct) and grade34_ae_pct > 0:
        if (grade34_ae_pct / 100) > PHASE1_AE_GRADE34_THRESHOLD:
            score -= 35
            notes.append(f"Grade 3/4 AE rate ({grade34_ae_pct:.1f}%) exceeds safety margin")
        else:
            notes.append(f"Grade 3/4 AE rate ({grade34_ae_pct:.1f}%) within acceptable margin")
    else:
        score -= 20
        notes.append("Grade 3/4 AE data sparse/not extracted — penalty applied for safety blindspot")
    return max(0, score), notes, "n/a"

def score_phase2(row):
    score = 100
    notes = []
    orr_pct = row.get("ORR_Pct")
    if orr_pct is not None and not pd.isna(orr_pct) and orr_pct > 0:
        notes.append(f"ORR reported: {orr_pct}%")
    else:
        score -= 30
        notes.append("ORR data not explicitly isolated from early summary records")

    assessment_method = row.get("ResponseAssessmentMethod")
    if assessment_method == "BICR":
        confidence = "high"
        notes.append("BICR verified — increased confidence footprint")
    else:
        confidence = "medium"
        score -= 10
        notes.append("Investigator-assessed only — modest verification discount applied")
    return max(0, score), notes, confidence

def score_phase3(row):
    score = 100
    notes = []
    hr = row.get("HazardRatio")
    ci_upper = row.get("HR_CI_Upper")
    if hr is not None and not pd.isna(hr) and hr > 0:
        if ci_upper is not None and not pd.isna(ci_upper) and ci_upper >= 1.0:
            score -= 40
            notes.append(f"HR 95% CI upper bound ({ci_upper}) crosses/touches 1.0 — high non-superiority risk")
        elif hr <= PHASE3_HR_EXCELLENT_THRESHOLD:
            notes.append(f"HR ({hr:.2f}) <= {PHASE3_HR_EXCELLENT_THRESHOLD} — excellent clinical efficacy signature")
        else:
            score -= 10
            notes.append(f"HR ({hr:.2f}) shows positive trend but subtle separation")
    else:
        score -= 30
        notes.append("Hazard Ratio missing from baseline results payload")

    enrollment = row.get("EnrollmentCount")
    if enrollment is not None and not pd.isna(enrollment):
        if enrollment < PHASE3_MIN_ENROLLMENT:
            score -= 25
            notes.append(f"Enrollment ({enrollment}) below sample size floor ({PHASE3_MIN_ENROLLMENT})")
        else:
            notes.append("Enrollment matches confirmatory statistical power expectations")
    return max(0, score), notes, "n/a"

# ------------------------------------------------------------
# ROBUST MULTI-PATH EXTROLLER
# ------------------------------------------------------------
def aggressive_v2_parser(raw_studies):
    records = []

    for study in raw_studies:
        protocol = study.get("protocolSection", {})
        results = study.get("resultsSection", {})

        nct_id = protocol.get("identificationModule", {}).get("nctId", "Unknown")
        title = protocol.get("identificationModule", {}).get("officialTitle", "")

        # --- PATHWAY A: ROBUST PHASE STRIPPING ---
        # Strategy: Scan multiple possible structural domains for Phase tags
        phases_list = protocol.get("statusModule", {}).get("phases", [])
        if not phases_list:
            # Backup path inside designModule
            phases_list = protocol.get("designModule", {}).get("designInfo", {}).get("phases", [])

        phase_str = "PHASE NOT SPECIFIED"
        if phases_list:
            # Squeeze and combine if it's a phase 1/2 crossover trial
            phase_str = "/".join(phases_list).upper()
        else:
            # Fallback text matching on title string if metadata is structurally broken
            title_upper = title.upper()
            if "PHASE 3" in title_upper or "PHASE III" in title_upper: phase_str = "PHASE3"
            elif "PHASE 2" in title_upper or "PHASE II" in title_upper: phase_str = "PHASE2"
            elif "PHASE 1" in title_upper or "PHASE I" in title_upper: phase_str = "PHASE1"

        status = protocol.get("statusModule", {}).get("overallStatus", "")
        enrollment = protocol.get("designModule", {}).get("enrollmentInfo", {}).get("count")
        sponsor = protocol.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {}).get("name", "")

        # --- PATHWAY B: SAFETY PARSING ---
        grade34_ae_pct = None
        other_aes = results.get("adverseEventsModule", {}).get("otherAdverseEvents", [])
        for ae in other_aes:
            term = ae.get("term", "").lower()
            if "grade 3" in term or "grade 4" in term or "severe" in term:
                stats = ae.get("stats", [{}])[0]
                num_affected = stats.get("numAffected", 0)
                if enrollment and enrollment > 0:
                    grade34_ae_pct = max(grade34_ae_pct or 0.0, (num_affected / enrollment) * 100)

        # --- PATHWAY C: EFFICACY / RECIST PARSING ---
        orr_pct = None
        assessment_method = "Investigator"
        outcome_measures = results.get("outcomeMeasuresModule", {}).get("outcomeMeasures", [])
        for outcome in outcome_measures:
            t_low = outcome.get("title", "").lower()
            d_low = outcome.get("description", "").lower()
            if any(x in t_low for x in ["overall response rate", "orr", "objective response", "tumor response"]):
                classes = outcome.get("classes", [{}])
                if classes and classes[0].get("categories"):
                    measurements = classes[0]["categories"][0].get("measurements", [{}])
                    if measurements:
                        try: orr_pct = float(measurements[0].get("value", 0))
                        except: pass
            if any(x in d_low for x in ["bicr", "independent central review", "blinded independent"]):
                assessment_method = "BICR"

        # --- PATHWAY D: STATISTICAL MATH PARSING ---
        hazard_ratio = None
        hr_ci_upper = None
        for outcome in outcome_measures:
            for analysis in outcome.get("analyses", []):
                method = analysis.get("statisticalMethod", "").lower()
                if "hazard ratio" in method or "hr" in method:
                    try:
                        hazard_ratio = float(analysis.get("paramValue", 0))
                        hr_ci_upper = float(analysis.get("ciUpperLimit", 0))
                    except: pass

        # Structural Row Assembly
        row_data = {
            "NCTId": nct_id,
            "Phase": phase_str,
            "EnrollmentCount": enrollment,
            "Grade34_AE_Pct": grade34_ae_pct,
            "ORR_Pct": orr_pct,
            "ResponseAssessmentMethod": assessment_method,
            "HazardRatio": hazard_ratio,
            "HR_CI_Upper": hr_ci_upper
        }

        # --- THE DYNAMIC PIPELINE ROUTER ---
        p = row_data["Phase"]
        if "PHASE1" in p or "PHASE 1" in p:
            sc, nt, cf = score_phase1(row_data)
        elif "PHASE2" in p or "PHASE 2" in p:
            sc, nt, cf = score_phase2(row_data)
        elif "PHASE3" in p or "PHASE 3" in p:
            sc, nt, cf = score_phase3(row_data)
        else:
            # Fallback if it's still unclassified
            sc, nt, cf = None, ["Phase structural type unknown — system skipped verification"], "not_scored"

        row_data["TrustScore"] = sc
        row_data["TrustScore_Notes"] = nt
        row_data["TrustScore_Confidence"] = cf
        row_data["LeadSponsor"] = sponsor
        records.append(row_data)

    return pd.DataFrame(records)

# ------------------------------------------------------------
# PIPELINE INVOCATION AND EXECUTION
# ------------------------------------------------------------
print("🚀 Triggering Deep Aggressive Phase Mapping Engine...")
full_scored_df = aggressive_v2_parser(all_raw_studies)

# Master Deduplication
df = full_scored_df.drop_duplicates(subset=["NCTId"]).reset_index(drop=True)

print(f"⚡ Success! {len(df)} distinct clinical data assets successfully structured.")
print("\n--- SYNCHRONIZED SCORING MATRIX ---")
display(df[["NCTId", "Phase", "TrustScore", "TrustScore_Confidence"]].head(15))

🚀 Triggering Deep Aggressive Phase Mapping Engine...
⚡ Success! 63 distinct clinical data assets successfully structured.

--- SYNCHRONIZED SCORING MATRIX ---


,NCTId,Phase,TrustScore,TrustScore_Confidence
0,NCT06024174,PHASE1,80.0,n/a
1,NCT05840510,PHASE1,80.0,n/a
2,NCT05853575,PHASE NOT SPECIFIED,NaN,not_scored
3,NCT05578092,PHASE1,80.0,n/a
4,NCT05609578,PHASE2,60.0,medium
5,NCT06497556,PHASE3,70.0,n/a
6,NCT06130254,PHASE1,80.0,n/a
7,NCT06801418,PHASE1,80.0,n/a
8,NCT05924152,PHASE1,80.0,n/a
9,NCT05673187,PHASE2,60.0,medium


In [18]:
# Save the structured master engine data into a high-grade JSON Knowledge Object file
df.to_json("kras_g12c_knowledge_objects.json", orient="records", indent=4)
print("📥 Success, CTO! 'kras_g12c_knowledge_objects.json' has been baked and saved in your Colab files sidebar!")

📥 Success, CTO! 'kras_g12c_knowledge_objects.json' has been baked and saved in your Colab files sidebar!


In [17]:
# ==============================================================================
# PRODUCTION-GRADE MULTI-FACTOR REGULATORY SCORING ENGINE (V3-ENTERPRISE)
# Optimized for Big Pharma Pitching — Implements dynamic weights and clinical constants.
# ==============================================================================

import pandas as pd
import numpy as np

class KRASRegulatoryScoringEngine:
    def __init__(self):
        # Universal Corporate Constants
        self.PHASE3_MIN_N = 200
        self.PHASE2_MIN_N = 60
        self.ALPHA_CRITICAL = 0.05

    def calculate_enterprise_score(self, row):
        """
        Executes a multi-factor regulatory matrix evaluation using matrix-inspired weights.
        """
        # Baseline Raw Metrics Extraction
        phase = str(row.get("Phase", "UNKNOWN")).upper()
        n = row.get("EnrollmentCount", 0)
        n = 0 if pd.isna(n) else int(n)

        orr = row.get("ORR_Pct")
        orr = None if pd.isna(orr) else float(orr)

        hr = row.get("HazardRatio")
        hr = None if pd.isna(hr) else float(hr)

        hr_upper = row.get("HR_CI_Upper")
        hr_upper = None if pd.isna(hr_upper) else float(hr_upper)

        ae_pct = row.get("Grade34_AE_Pct")
        ae_pct = 0.0 if pd.isna(ae_pct) else float(ae_pct)

        method = str(row.get("ResponseAssessmentMethod", "Investigator")).upper()
        sponsor = str(row.get("LeadSponsor", "")).lower()
        title = str(row.get("OfficialTitle", "")).lower()

        # ----------------------------------------------------------------------
        # LAYER 1: DYNAMIC WEIGHT ALLOCATION (Based on Clinical Phase Realities)
        # ----------------------------------------------------------------------
        if "PHASE1" in phase:
            w_p, w_e, w_s, w_r = 0.10, 0.10, 0.60, 0.20  # Phase 1 is 60% Safety-driven
        elif "PHASE2" in phase:
            w_p, w_e, w_s, w_r = 0.20, 0.40, 0.20, 0.20  # Phase 2 is 40% Efficacy/ORR-driven
        elif "PHASE3" in phase:
            w_p, w_e, w_s, w_r = 0.40, 0.30, 0.10, 0.20  # Phase 3 is 40% Statistical Power/HR-driven
        else:
            w_p, w_e, w_s, w_r = 0.25, 0.25, 0.25, 0.25  # Standard default normalization

        # ----------------------------------------------------------------------
        # LAYER 2: FACTOR SUB-MATRICES COMPUTATION (0 - 100 Scales)
        # ----------------------------------------------------------------------

        # --- 2.1 Statistical Power Vector (P) ---
        score_P = 100
        if "PHASE3" in phase and n < self.PHASE3_MIN_N:
            score_P -= 30  # High penalty if Phase 3 sample sizing is underpowered
        elif "PHASE2" in phase and n < self.PHASE2_MIN_N:
            score_P -= 15

        if hr_upper is not None:
            if hr_upper >= 1.0:
                score_P -= 40  # Massive statistical risk if crossing the line of unity
            elif hr_upper > 0.85:
                score_P -= 10  # Partial penalty for wide/unstable confidence intervals

        # --- 2.2 Clinical Efficacy Vector (E) ---
        score_E = 100
        # Dynamic Line of Therapy & Endpoint Extractor
        is_late_line = any(x in title for x in ["previously treated", "second line", "2l", "3l", "refractory"])

        if orr is not None:
            if is_late_line:
                if orr < 30.0: score_E -= 20  # Grace path: late-line thresholds are lower in oncology
            else:
                if orr < 55.0: score_E -= 35  # Strict path: 1L frontline requires superior ORR footprint
        else:
            score_E -= 30  # Data missing penalty

        # Hard Endpoint Check (OS vs PFS/ORR)
        if "overall survival" in title or "os" in title:
            score_E = min(100, score_E + 10)  # Institutional Premium Bonus for hard survival data

        # --- 2.3 Safety & Toxicity Vector (S) ---
        # Note: This returns a penalty footprint (higher AE dense = higher penalty value)
        penalty_S = 0
        if ae_pct > 40.0:
            penalty_S += 40  # Alarming toxic envelope
        elif ae_pct > 20.0:
            penalty_S += 15  # Acceptable standard oncology toxicity

        if "combination" in title or "plus" in title:
            penalty_S = max(0, penalty_S - 5)  # Combination therapies are expected to be more toxic

        # --- 2.4 Regulatory Quality Vector (R) ---
        score_R = 100
        if "BICR" in method:
            score_R += 15  # Blinded Independent Review execution standard bonus

        # Sponsor Multiplier
        if any(x in sponsor for x in ["university", "institute", "cancer center", "nih", "nci"]):
            score_R = min(100, score_R + 10)  # Academic/Cooperative groups carry lower commercial bias risk

        # Active Control Arm Check
        if "vs" in title or "compared to" in title or "docetaxel" in title:
            score_R = min(100, score_R + 10)  # High marks for running against a real active control

        # ----------------------------------------------------------------------
        # LAYER 3: MATRICES SYNTHESIS & SCORING BOUNDS CONSTRAINTS
        # ----------------------------------------------------------------------
        final_score = (w_p * score_P) + (w_e * score_E) - (w_s * penalty_S) + (w_r * score_R)
        final_score = max(0.0, min(100.0, final_score)) # Hard industrial engineering safe bounds

        # Traceability Logs for Corporate Transparency
        audit_trail = (
            f"Weights[P:{w_p}, E:{w_e}, S:{w_s}, R:{w_r}] | "
            f"Vectors[P_Score:{score_P}, E_Score:{score_E}, S_Penalty:{penalty_S}, R_Score:{score_R}] | "
            f"LoT:{'Late-Line' if is_late_line else 'Front-Line'}"
        )

        return round(final_score, 1), audit_trail

# ------------------------------------------------------------------------------
# PIPELINE TRIGGER ENGINE
# ------------------------------------------------------------------------------
engine = KRASRegulatoryScoringEngine()

print("🔮 Initializing Enterprise Multi-Factor Scoring Pipeline Matrix...")
enterprise_results = df.apply(engine.calculate_enterprise_score, axis=1)

# Map into fresh production columns inside your global DataFrame
df["Enterprise_TrustScore"] = enterprise_results.apply(lambda x: x[0])
df["Architect_Audit_Trail"] = enterprise_results.apply(lambda x: x[1])

print("⚡ Processing Complete. Displaying high-fidelity structured outcomes:\n")
display(df[["NCTId", "Phase", "EnrollmentCount", "Enterprise_TrustScore", "Architect_Audit_Trail"]].head(10))

🔮 Initializing Enterprise Multi-Factor Scoring Pipeline Matrix...
⚡ Processing Complete. Displaying high-fidelity structured outcomes:



,NCTId,Phase,EnrollmentCount,Enterprise_TrustScore,Architect_Audit_Trail
0,NCT06024174,PHASE1,5,40.0,"Weights[P:0.1, E:0.1, S:0.6, R:0.2] | Vectors[..."
1,NCT05840510,PHASE1,6,37.0,"Weights[P:0.1, E:0.1, S:0.6, R:0.2] | Vectors[..."
2,NCT05853575,PHASE NOT SPECIFIED,200,67.5,"Weights[P:0.25, E:0.25, S:0.25, R:0.25] | Vect..."
3,NCT05578092,PHASE1,64,37.0,"Weights[P:0.1, E:0.1, S:0.6, R:0.2] | Vectors[..."
4,NCT05609578,PHASE2,90,68.0,"Weights[P:0.2, E:0.4, S:0.2, R:0.2] | Vectors[..."
5,NCT06497556,PHASE3,338,81.0,"Weights[P:0.4, E:0.3, S:0.1, R:0.2] | Vectors[..."
6,NCT06130254,PHASE1,1,37.0,"Weights[P:0.1, E:0.1, S:0.6, R:0.2] | Vectors[..."
7,NCT06801418,PHASE1,160,37.0,"Weights[P:0.1, E:0.1, S:0.6, R:0.2] | Vectors[..."
8,NCT05924152,PHASE1,16,37.0,"Weights[P:0.1, E:0.1, S:0.6, R:0.2] | Vectors[..."
9,NCT05673187,PHASE2,68,68.0,"Weights[P:0.2, E:0.4, S:0.2, R:0.2] | Vectors[..."


In [19]:
# Selyadong JSON export code para sa V3 Enterprise Database
df.to_json("kras_g12c_knowledge_objects.json", orient="records", indent=4)
print("📥 SUCCESS, CTO! Ang 'kras_g12c_knowledge_objects.json' ay matagumpay na na-bake!")
print("-> Naglalaman ito ng 63 unique trials na may Dynamic Multi-Factor Trust Scores.")
print("-> Pwede mo na itong i-download mula sa kaliwang sidebar at i-send kay Kiefer!")

📥 SUCCESS, CTO! Ang 'kras_g12c_knowledge_objects.json' ay matagumpay na na-bake!
-> Naglalaman ito ng 63 unique trials na may Dynamic Multi-Factor Trust Scores.
-> Pwede mo na itong i-download mula sa kaliwang sidebar at i-send kay Kiefer!
